# Importing the Libraries

In [23]:
import re
import pandas as pd
from datasets import load_dataset

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense


# Load Dataset from Hugging Face



In [8]:
from datasets import load_dataset
dataset = load_dataset("wangrongsheng/ag_news")

# Seperate train and test dataset

In [27]:
train_data = dataset["train"]
test_data = dataset["test"]

In [28]:
print(dataset)  # here printing wangrongsheng/ag_news dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [32]:
train_df = train_data.to_pandas() # Convert to pandas dataframes
test_df = test_data.to_pandas()
print("\nTraining Data")
print(train_df.head())    # here it display top 5 data

print("\nTesting Data")
print(test_df.head())

print(train_df.head())


Training Data
                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2

Testing Data
                                                text  label
0  Fears for T N pension after talks Unions repre...      2
1  The Race is On: Second Private Team Sets Launc...      3
2  Ky. Company Wins Grant to Study Peptides (AP) ...      3
3  Prediction Unit Helps Forecast Wildfires (AP) ...      3
4  Calif. Aims to Limit Farm-Related Smog (AP) AP...      3
                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Hal

# Cleaning the Text

In [14]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # re stands for Regular Expressions, a Python module used for pattern matching.
    return text

train_df["text"] = train_df["text"].apply(clean_text)
test_df["text"] = test_df["text"].apply(clean_text)


# Extract Text and Labels

In [15]:
X_train_text = train_df["text"]
X_test_text = test_df["text"]

y_train = train_df["label"]
y_test = test_df["label"]

# Tokenization

In [16]:
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size)

tokenizer.fit_on_texts(X_train_text)

X_train = tokenizer.texts_to_sequences(X_train_text)
X_test = tokenizer.texts_to_sequences(X_test_text)


# Padding

In [17]:
max_length = 50

X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding="post",
    truncating="post"
)


# One Hot Encoded Labels

In [19]:
num_classes = 4

y_train = to_categorical(y_train, num_classes=num_classes)
y_test = to_categorical(y_test, num_classes=num_classes)

# Checking Shapes

In [20]:

print("\nTraining Data Shape :", X_train.shape)
print("Testing Data Shape  :", X_test.shape)
print("\nTraining Labels Shape :", y_train.shape)
print("Testing Labels Shape  :", y_test.shape)


Training Data Shape : (120000, 50)
Testing Data Shape  : (7600, 50)

Training Labels Shape : (120000, 4, 4)
Testing Labels Shape  : (7600, 4, 4)


# Sample Output

In [21]:
print("\nFirst Tokenized Sequence:")
print(X_train[0])

print("\nFirst One-Hot Label:")
print(y_train[0])


First Tokenized Sequence:
[ 391  324 1525   99   54    1  812   23   23  391 1988    4   34 3893
  737  295    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]

First One-Hot Label:
[[1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]]


# Build the model with an explicit input length

In [49]:
model = Sequential([ # Sequential creates a neural network by stacking layers one after another.
    Embedding(             # The Embedding layer converts each word ID into a dense vector of numbers.
        input_dim=10000,
        output_dim=128,
        input_length=50
    ),
    SimpleRNN(64),              # It reads the sentence one word at a time.
    Dense(4, activation="softmax")  # Your AG News dataset has 4 multiple classes:
])

model.compile(           # This prepares the model for training by specifying how it should learn.
    optimizer="adam",     # The optimizer updates the model's weights to reduce prediction errors.
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary() # Model Summary

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# Check my data

In [45]:
print("X_train shape:", X_train.shape)
print("X_train dtype:", X_train.dtype)

print("y_train shape:", y_train.shape)
print("y_train dtype:", y_train.dtype)

print("First X_train sample:")
print(X_train[0])

print("First y_train sample:")
print(y_train[0])

X_train shape: (120000, 50)
X_train dtype: int32
y_train shape: (120000, 4)
y_train dtype: float64
First X_train sample:
[ 391  324 1525   99   54    1  812   23   23  391 1988    4   34 3893
  737  295    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]
First y_train sample:
[0. 0. 1. 0.]


# Train the Model

In [48]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 49s 33ms/step - accuracy: 0.9337 - loss: 0.2275 - val_accuracy: 0.8800 - val_loss: 0.3947
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 48s 32ms/step - accuracy: 0.9493 - loss: 0.1718 - val_accuracy: 0.8739 - val_loss: 0.4377
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 83s 32ms/step - accuracy: 0.9566 - loss: 0.1502 - val_accuracy: 0.8735 - val_loss: 0.4476
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 49s 33ms/step - accuracy: 0.9584 - loss: 0.1414 - val_accuracy: 0.8628 - val_loss: 0.5173
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 48s 32ms/step - accuracy: 0.9553 - loss: 0.1537 - val_accuracy: 0.8768 - val_loss: 0.4963


# Evaluate the Models Loss and Accuracy

In [47]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss :", loss)
print("Test Accuracy :", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.8486 - loss: 0.4695
Test Loss : 0.46945056319236755
Test Accuracy : 0.8485526442527771
